## Imports

In [71]:
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

import tensorflow as tf
import numpy as np
import kagglehub
import cv2
import os
import random
import glob

try:
    from google.colab import drive
    COLAB = True
except ImportError:
    COLAB = False

In [72]:
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

## Config

In [73]:
IMG_HEIGHT = 128
IMG_WIDTH = 128
BATCH_SIZE = 32
EPOCHS = 10
NUM_CLASSES = 10

## Utils

In [74]:
def load_and_preprocess_data(train_dir, val_dir):
    train_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir, image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE, label_mode='categorical'
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        val_dir, image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE, label_mode='categorical'
    )

    return train_ds, val_ds

In [75]:
def build_recognition_model(num_classes):
    model = models.Sequential()
    model.add(layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)))

    # Normalization
    model.add(layers.Rescaling(1./255))

    # Data Augmentation
    model.add(layers.RandomFlip("horizontal"))
    model.add(layers.RandomRotation(0.1))
    model.add(layers.RandomZoom(0.1))

    # Core Convolutional Blocks
    model.add(layers.Conv2D(32, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))

    model.add(layers.Conv2D(64, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))

    model.add(layers.Conv2D(128, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2)))

    # Pooling and Dense Layers
    model.add(layers.GlobalAveragePooling2D())
    model.add(layers.Dense(256, activation='relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(num_classes, activation='softmax'))

    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

In [84]:
def predict_real_time(model, image_path, class_names):
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))

    # Expand dimensions to match batch format: (1, 128, 128, 3)
    img_array = np.expand_dims(img_resized, axis=0)

    predictions = model.predict(img_array, verbose=0)
    predicted_class_idx = np.argmax(predictions[0])
    confidence = predictions[0][predicted_class_idx]

    return class_names[predicted_class_idx], confidence

## Object Detection

In [77]:
dataset_path = kagglehub.dataset_download("moltean/fruits")
# dataset_path = kagglehub.dataset_download("nunenuh/pytorch-challange-flower-dataset")
print("Path to dataset files:", dataset_path)

Using Colab cache for faster access to the 'fruits' dataset.
Path to dataset files: /kaggle/input/fruits


In [78]:
base_dir = os.path.join(dataset_path, 'fruits-360_100x100', 'fruits-360')
train_dir = os.path.join(base_dir, "Training")
test_dir = os.path.join(base_dir, "Test")

# print(f"Base directory: {base_dir}")
# print(f"Training images ready at: {train_dir}")
# print(f"Test images ready at: {test_dir}")

In [79]:
print("Loading Data...")
raw_train_ds = tf.keras.utils.image_dataset_from_directory(train_dir)
class_names = raw_train_ds.class_names
NUM_CLASSES = len(class_names)

train_ds, val_ds = load_and_preprocess_data(train_dir, test_dir)

Loading Data...
Found 135071 files belonging to 257 classes.
Found 135071 files belonging to 257 classes.
Found 45008 files belonging to 257 classes.


In [80]:
print("Building Model...")
model = build_recognition_model(NUM_CLASSES)
model.summary()

Building Model...


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_4 (Rescaling)         │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip_2 (RandomFlip)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation_2               │ (None, 128, 128, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom_2 (RandomZoom)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_12 (Conv2D)              │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_12          │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_13          │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_13 (MaxPooling2D) │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_14          │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_14 (MaxPooling2D) │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 257)            │        66,049 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 193,217 (754.75 KB)

 Trainable params: 192,769 (753.00 KB)

 Non-trainable params: 448 (1.75 KB)

In [81]:
print("Training Model...")
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stopping]
)

Training Model...
Epoch 1/10
4221/4221 ━━━━━━━━━━━━━━━━━━━━ 218s 51ms/step - accuracy: 0.6283 - loss: 1.2943 - val_accuracy: 0.8690 - val_loss: 0.7440
Epoch 2/10
4221/4221 ━━━━━━━━━━━━━━━━━━━━ 226s 54ms/step - accuracy: 0.8954 - loss: 0.3048 - val_accuracy: 0.9235 - val_loss: 0.7068
Epoch 3/10
4221/4221 ━━━━━━━━━━━━━━━━━━━━ 218s 52ms/step - accuracy: 0.9365 - loss: 0.1828 - val_accuracy: 0.9141 - val_loss: 0.7141
Epoch 4/10
4221/4221 ━━━━━━━━━━━━━━━━━━━━ 261s 52ms/step - accuracy: 0.9533 - loss: 0.1333 - val_accuracy: 0.9213 - val_loss: 0.6993
Epoch 5/10
4221/4221 ━━━━━━━━━━━━━━━━━━━━ 217s 51ms/step - accuracy: 0.9616 - loss: 0.1078 - val_accuracy: 0.9644 - val_loss: 0.5664
Epoch 6/10
4221/4221 ━━━━━━━━━━━━━━━━━━━━ 217s 51ms/step - accuracy: 0.9681 - loss: 0.0901 - val_accuracy: 0.9442 - val_loss: 0.6034
Epoch 7/10
4221/4221 ━━━━━━━━━━━━━━━━━━━━ 217s 51ms/step - accuracy: 0.9717 - loss: 0.0795 - val_accuracy: 0.9533 - val_loss: 0.6947
Epoch 8/10
4221/4221 ━━━━━━━━━━━━━━━━━━━━ 217s 51ms

In [82]:
print("Saving Model...")

MODEL = 'fruits_moodel_v3.keras'
# MODEL = 'flowers_model.keras'

if COLAB:
    drive.mount('/content/drive')
    model_path = f'/content/drive/MyDrive/{MODEL}'
else:
    model_path = MODEL

model.save(model_path)
print(f"Model successfully saved to: {model_path}")

Saving Model...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model successfully saved to: /content/drive/MyDrive/fruits_moodel_v3.keras


In [85]:
print("Predictions on Random Test Images...")

all_test_images = glob.glob(os.path.join(test_dir, "*", "*.jpg"))
random_samples = random.sample(all_test_images, 50)

for img_path in random_samples:
    # Extract the true label
    true_label = os.path.basename(os.path.dirname(img_path))

    identity, conf = predict_real_time(model, img_path, class_names)

    print(f"True: {true_label:<15} | Predicted: {identity:<15} | Confidence: {conf:.2f}")

Predictions on Random Test Images...
True: Zucchini Green 1 | Predicted: Zucchini Green 1 | Confidence: 0.50
True: Pepper Red 1    | Predicted: Pepper Red 1    | Confidence: 1.00
True: Mandarine 1     | Predicted: Mandarine 1     | Confidence: 1.00
True: Apple 14        | Predicted: Apple 14        | Confidence: 1.00
True: Avocado Black 2 | Predicted: Avocado Black 2 | Confidence: 1.00
True: Cucumber 10     | Predicted: Cucumber 10     | Confidence: 1.00
True: Pear 7          | Predicted: Pear 7          | Confidence: 1.00
True: Grapefruit Pink 1 | Predicted: Grapefruit Pink 1 | Confidence: 1.00
True: Grape Blue 1    | Predicted: Grape Blue 1    | Confidence: 1.00
True: Tomato Heart 1  | Predicted: Tomato Heart 1  | Confidence: 1.00
True: Cherry Rainier 3 | Predicted: Cherry Rainier 3 | Confidence: 1.00
True: Mangostan 1     | Predicted: Mangostan 1     | Confidence: 1.00
True: Gooseberry 1    | Predicted: Gooseberry 1    | Confidence: 1.00
True: Apricot 1       | Predicted: Apricot 1 